# Phase 2: Feature Extraction & Engineering
### Multi-Source Entity Resolution (Sources 1, 2, 3 - Train & Test)

**Goal:** Turn the normalized (+ raw) fields from Phase 1 into rich numeric and categorical features that candidate blocking and similarity scoring can utilize.

**Scope:**
- Applies to all 6 files from Phase 1's output (`data/processed/phase1/`):
  - `train_source1_normalized.tsv`, `train_source2_normalized.tsv`, `train_source3_normalized.tsv`
  - `test_source1_normalized.tsv`, `test_source2_normalized.tsv`, `test_source3_normalized.tsv`
- **Feature Categories:**
  1. **Name Features (`name_features.py`):**
     - Per-record: `name_char_len`, `name_word_count`, `name_tokens_sorted`, `name_first_token`, `name_last_token`, `name_soundex`
     - Pairwise Engine: Jaccard Similarity (token overlap), Levenshtein Distance & Normalized Similarity, TF-IDF Cosine Similarity
  2. **Address Features (`address_features.py`):**
     - Per-record: `addr_char_len`, `addr_word_count`, `addr_tokens_sorted`, `addr_first_token`, `addr_has_landmark`
     - Pairwise Engine: Address token Jaccard, address string Levenshtein
  3. **Geographic Features (`geographic_features.py`):**
     - PIN/ZIP extraction (US 5-digit, India 6-digit, France 5-digit)
     - Postal code prefix (first 2-3 digits) and Region mapping
     - Landmark reference indicator ("near", "opp", "behind")
  4. **Country Features (`country_features.py`):**
     - Canonical country label and address pattern tags (`addr_pattern_us`, `addr_pattern_india`, `addr_pattern_france`)
     - Match/mismatch country comparator
  5. **Completeness Features (`completeness_features.py`):**
     - Missingness flags (`is_name_empty`, `is_addr_empty`, `is_country_empty`, `is_postal_empty`)
     - Completeness score (0.0 to 1.0) and sparsity tier (`HIGH`, `MEDIUM`, `LOW`)
  6. **Cross-Field Features (`cross_field_features.py`):**
     - Country vs postal code consistency flag
     - Name token overlap in address
     - Joint representation: `combined_name_address`
  7. **Feature Builder & Streaming Batch Pipeline:**
     - Orchestrates features into one table and writes to `data/processed/phase2/`
  8. **Pairwise Similarity Demonstration on Ground Truth Matches:**
     - Evaluates true matches using Jaccard, Levenshtein, and TF-IDF Cosine similarity.


In [1]:
import os
import sys
import re
import math
import csv
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional, Any, Set

# Set project roots
ROOT_DIR = Path("..").resolve() if Path("..").resolve().name == "AMAZON_ML_CHALLENGE" else Path(".").resolve()
PHASE1_DIR = ROOT_DIR / "data" / "processed" / "phase1"
PHASE2_DIR = ROOT_DIR / "data" / "processed" / "phase2"
PHASE2_DIR.mkdir(parents=True, exist_ok=True)

GT_FILE = ROOT_DIR / "student_resource" / "dataset" / "train" / "train_ground_truth.tsv"

print(f"Project Root: {ROOT_DIR}")
print(f"Phase 1 Input Dir: {PHASE1_DIR} (Exists: {PHASE1_DIR.exists()})")
print(f"Phase 2 Output Dir: {PHASE2_DIR}")
print(f"Ground Truth File: {GT_FILE} (Exists: {GT_FILE.exists()})")


Project Root: D:\Amazon ML Challenge\AMAZON_ML_CHALLENGE
Phase 1 Input Dir: D:\Amazon ML Challenge\AMAZON_ML_CHALLENGE\data\processed\phase1 (Exists: True)
Phase 2 Output Dir: D:\Amazon ML Challenge\AMAZON_ML_CHALLENGE\data\processed\phase2
Ground Truth File: D:\Amazon ML Challenge\AMAZON_ML_CHALLENGE\student_resource\dataset\train\train_ground_truth.tsv (Exists: True)


## Step 1: Phonetic & String Utilities (Soundex & String Hashing)
Phonetic encoding enables candidate generation to catch phonetically identical but misspelled names (e.g. *Phillips* vs *Philips*, *Smith* vs *Smyth*).


In [2]:
def compute_soundex(name: str) -> str:
    """Computes standard American Soundex code for a given token."""
    if not name:
        return "0000"
    name = re.sub(r"[^A-Za-z]", "", name.upper())
    if not name:
        return "0000"
    
    first_letter = name[0]
    mapping = {
        'B': '1', 'F': '1', 'P': '1', 'V': '1',
        'C': '2', 'G': '2', 'J': '2', 'K': '2', 'Q': '2', 'S': '2', 'X': '2', 'Z': '2',
        'D': '3', 'T': '3',
        'L': '4',
        'M': '5', 'N': '5',
        'R': '6'
    }
    
    encoded = [first_letter]
    prev = mapping.get(first_letter, "0")
    
    for char in name[1:]:
        digit = mapping.get(char, "0")
        if digit != "0" and digit != prev:
            encoded.append(digit)
            if len(encoded) == 4:
                break
        prev = digit
        
    while len(encoded) < 4:
        encoded.append("0")
        
    return "".join(encoded[:4])

# Test Soundex
print("Soundex tests:")
for test in ["Phillips", "Philips", "Smith", "Smyth", "Barbershop", "Hospital"]:
    print(f"  {test:12} -> {compute_soundex(test)}")


Soundex tests:
  Phillips     -> P412
  Philips      -> P412
  Smith        -> S530
  Smyth        -> S530
  Barbershop   -> B616
  Hospital     -> H213


## Step 2: Pairwise Similarity Algorithms (Jaccard, Levenshtein, TF-IDF Cosine)
As required, we implement three core similarity measures:
1. **Jaccard Similarity (Token Overlap):** $|A \cap B| / |A \cup B|$
2. **Levenshtein Distance & Normalized Similarity:** $1 - (\text{dist} / \max(\text{len}_A, \text{len}_B))$
3. **TF-IDF Cosine Similarity:** Term frequency dot product normalized by $L_2$ vectors (order-invariant matching).


In [3]:
def jaccard_similarity(s1: str, s2: str) -> float:
    """Calculates token overlap Jaccard similarity between two strings."""
    if not s1 and not s2:
        return 1.0
    if not s1 or not s2:
        return 0.0
    tokens1 = set(s1.lower().split())
    tokens2 = set(s2.lower().split())
    if not tokens1 and not tokens2:
        return 1.0
    if not tokens1 or not tokens2:
        return 0.0
    return len(tokens1 & tokens2) / len(tokens1 | tokens2)

def levenshtein_distance(s1: str, s2: str) -> int:
    """Calculates Levenshtein edit distance between two strings with O(min(M, N)) space."""
    if s1 == s2:
        return 0
    if not s1:
        return len(s2)
    if not s2:
        return len(s1)
        
    if len(s1) > len(s2):
        s1, s2 = s2, s1
        
    prev = list(range(len(s1) + 1))
    for j, c2 in enumerate(s2, 1):
        curr = [j] + [0] * len(s1)
        for i, c1 in enumerate(s1, 1):
            cost = 0 if c1 == c2 else 1
            curr[i] = min(curr[i-1] + 1, prev[i] + 1, prev[i-1] + cost)
        prev = curr
    return prev[-1]

def levenshtein_similarity(s1: str, s2: str) -> float:
    """Converts Levenshtein distance into a normalized similarity score between 0.0 and 1.0."""
    max_len = max(len(s1), len(s2))
    if max_len == 0:
        return 1.0
    dist = levenshtein_distance(s1, s2)
    return max(0.0, 1.0 - (dist / max_len))

def tfidf_cosine_similarity(s1: str, s2: str) -> float:
    """Calculates TF-IDF cosine similarity between two strings without external dependencies."""
    if not s1 and not s2:
        return 1.0
    if not s1 or not s2:
        return 0.0
    
    tokens1 = Counter(s1.lower().split())
    tokens2 = Counter(s2.lower().split())
    
    all_tokens = set(tokens1.keys()) | set(tokens2.keys())
    if not all_tokens:
        return 1.0
        
    dot = sum(tokens1.get(t, 0) * tokens2.get(t, 0) for t in all_tokens)
    norm1 = math.sqrt(sum(v**2 for v in tokens1.values()))
    norm2 = math.sqrt(sum(v**2 for v in tokens2.values()))
    
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)

# Quick demonstration
s_a = "urology partners inc"
s_b = "urology partners corporation"
print("Pairwise Metric Demonstration:")
print(f"  String A: '{s_a}'")
print(f"  String B: '{s_b}'")
print(f"  - Jaccard Similarity (token overlap) : {jaccard_similarity(s_a, s_b):.4f}")
print(f"  - Levenshtein Similarity (edit dist)  : {levenshtein_similarity(s_a, s_b):.4f}")
print(f"  - TF-IDF Cosine Similarity            : {tfidf_cosine_similarity(s_a, s_b):.4f}")


Pairwise Metric Demonstration:
  String A: 'urology partners inc'
  String B: 'urology partners corporation'
  - Jaccard Similarity (token overlap) : 0.5000
  - Levenshtein Similarity (edit dist)  : 0.6429
  - TF-IDF Cosine Similarity            : 0.6667


## Step 3: Geographic Reference Rules & Postal Mappings
We extract:
- **US 5-digit ZIP** and map 2-digit prefix to US State/Region.
- **India 6-digit PIN** and map 2-digit prefix to Postal Circle/Region.
- **France 5-digit Postal Code** and map 2-digit prefix to French Department.


In [4]:
# Geographic prefix to Region mapping
US_ZIP_TO_REGION = {
    "01": "MA", "02": "MA", "03": "NH", "04": "ME", "05": "VT", "06": "CT", "07": "NJ", "08": "NJ",
    "10": "NY", "11": "NY", "12": "NY", "13": "NY", "14": "NY", "15": "PA", "16": "PA", "17": "PA", "18": "PA", "19": "PA",
    "20": "DC", "21": "MD", "22": "VA", "23": "VA", "24": "VA", "27": "NC", "28": "NC", "29": "SC",
    "30": "GA", "31": "GA", "32": "FL", "33": "FL", "37": "TN", "38": "TN", "39": "MS",
    "40": "KY", "41": "KY", "43": "OH", "44": "OH", "45": "OH", "46": "IN", "47": "IN", "48": "MI", "49": "MI",
    "50": "IA", "51": "IA", "52": "IA", "53": "WI", "54": "WI", "55": "MN", "56": "MN", "57": "SD", "58": "ND", "59": "MT",
    "60": "IL", "61": "IL", "62": "IL", "63": "MO", "64": "MO", "66": "KS", "67": "KS", "68": "NE", "69": "NE",
    "70": "LA", "71": "LA", "72": "AR", "73": "OK", "74": "OK", "75": "TX", "76": "TX", "77": "TX", "78": "TX", "79": "TX",
    "80": "CO", "81": "CO", "82": "WY", "83": "ID", "84": "UT", "85": "AZ", "87": "NM", "88": "NM", "89": "NV",
    "90": "CA", "91": "CA", "92": "CA", "93": "CA", "94": "CA", "95": "CA", "96": "CA", "97": "OR", "98": "WA", "99": "WA"
}

INDIA_PIN_TO_REGION = {
    "11": "Delhi", "12": "Haryana", "13": "Haryana", "14": "Punjab", "15": "Punjab", "16": "Punjab/Chandigarh",
    "17": "Himachal Pradesh", "18": "Jammu & Kashmir", "19": "Jammu & Kashmir",
    "20": "Uttar Pradesh", "21": "Uttar Pradesh", "22": "Uttar Pradesh", "23": "Uttar Pradesh", "24": "Uttar Pradesh",
    "25": "Uttar Pradesh", "26": "Uttar Pradesh", "27": "Uttar Pradesh", "28": "Uttar Pradesh",
    "30": "Rajasthan", "31": "Rajasthan", "32": "Rajasthan", "33": "Rajasthan", "34": "Rajasthan",
    "36": "Gujarat", "37": "Gujarat", "38": "Gujarat", "39": "Gujarat",
    "40": "Maharashtra", "41": "Maharashtra", "42": "Maharashtra", "43": "Maharashtra", "44": "Maharashtra",
    "45": "Madhya Pradesh", "46": "Madhya Pradesh", "47": "Madhya Pradesh", "48": "Madhya Pradesh", "49": "Chhattisgarh",
    "50": "Telangana/Andhra Pradesh", "51": "Andhra Pradesh", "52": "Andhra Pradesh", "53": "Andhra Pradesh",
    "56": "Karnataka", "57": "Karnataka", "58": "Karnataka", "59": "Karnataka",
    "60": "Tamil Nadu", "61": "Tamil Nadu", "62": "Tamil Nadu", "63": "Tamil Nadu", "64": "Tamil Nadu",
    "67": "Kerala", "68": "Kerala", "69": "Kerala",
    "70": "West Bengal", "71": "West Bengal", "72": "West Bengal", "73": "West Bengal", "74": "West Bengal",
    "75": "Odisha", "76": "Odisha", "77": "Odisha", "78": "Assam", "79": "North East",
    "80": "Bihar", "81": "Bihar", "82": "Bihar", "83": "Jharkhand", "84": "Bihar", "85": "Bihar"
}

FRANCE_POSTAL_TO_REGION = {
    "75": "Paris", "92": "Hauts-de-Seine", "93": "Seine-Saint-Denis", "94": "Val-de-Marne",
    "69": "Rhone", "13": "Bouches-du-Rhone", "33": "Gironde", "31": "Haute-Garonne", "59": "Nord"
}

RE_US_ZIP = re.compile(r"\b(\d{5})(?:-\d{4})?\b")
RE_INDIA_PIN = re.compile(r"\b([1-9]\d{5})\b|\b([1-9]\d{2}\s*\d{3})\b")
RE_LANDMARKS = re.compile(r"\b(near|opp|opposite|behind|beside|adjacent to|at|infront of)\b", re.IGNORECASE)

print("Postal mapping dictionaries compiled successfully!")


Postal mapping dictionaries compiled successfully!


## Step 4: Per-Record Feature Extraction Modules
We construct modular extractors for:
1. `extract_name_features`: character length, word count, sorted tokens, first token, soundex.
2. `extract_address_features`: character length, word count, sorted tokens, first token, landmark indicator.
3. `extract_geographic_features`: extracted postal code, postal prefix, inferred region.
4. `extract_country_features`: country pattern detection (US vs India vs France format).
5. `extract_completeness_features`: missingness flags, completeness score, sparsity tier.
6. `extract_cross_field_features`: postal-country consistency, name in address, combined text.


In [5]:
def extract_name_features(name_norm: str) -> Dict[str, Any]:
    """Extracts per-record lexical, token, and phonetic name features."""
    tokens = name_norm.split()
    char_len = len(name_norm)
    word_count = len(tokens)
    
    tokens_sorted = " ".join(sorted(tokens)) if tokens else ""
    first_token = tokens[0] if tokens else ""
    last_token = tokens[-1] if tokens else ""
    
    # Soundex on the first significant word (skipping common single letters)
    first_sig_token = next((t for t in tokens if len(t) > 1), first_token)
    name_soundex = compute_soundex(first_sig_token)
    
    return {
        "name_char_len": char_len,
        "name_word_count": word_count,
        "name_tokens_sorted": tokens_sorted,
        "name_first_token": first_token,
        "name_last_token": last_token,
        "name_soundex": name_soundex
    }

def extract_address_features(addr_norm: str, addr_raw: str) -> Dict[str, Any]:
    """Extracts per-record lexical, structural, and landmark address features."""
    tokens = addr_norm.replace(",", " ").split()
    char_len = len(addr_norm)
    word_count = len(tokens)
    
    tokens_sorted = " ".join(sorted(tokens)) if tokens else ""
    first_token = tokens[0] if tokens else ""
    
    # Landmark flag: checks both raw and normalized addresses
    has_landmark = 1 if (RE_LANDMARKS.search(addr_raw) or RE_LANDMARKS.search(addr_norm)) else 0
    
    return {
        "addr_char_len": char_len,
        "addr_word_count": word_count,
        "addr_tokens_sorted": tokens_sorted,
        "addr_first_token": first_token,
        "addr_has_landmark": has_landmark
    }

def extract_geographic_features(addr_raw: str, addr_norm: str, country_norm: str) -> Dict[str, Any]:
    """Extracts postal codes, postal prefixes, and infers region/state."""
    combined_addr = f"{addr_raw} {addr_norm}"
    postal_code = ""
    postal_prefix = ""
    inferred_region = ""
    
    country_lower = country_norm.lower()
    
    # 1. India PIN detection
    if "india" in country_lower or country_norm == "IN":
        pin_match = RE_INDIA_PIN.search(combined_addr)
        if pin_match:
            postal_code = re.sub(r"\s+", "", pin_match.group(0))
            postal_prefix = postal_code[:2]
            inferred_region = INDIA_PIN_TO_REGION.get(postal_prefix, "India-Other")
            
    # 2. US ZIP detection
    if not postal_code and ("us" in country_lower or country_norm in ["US", "USA"]):
        zip_match = RE_US_ZIP.search(combined_addr)
        if zip_match:
            postal_code = zip_match.group(1)
            postal_prefix = postal_code[:2]
            inferred_region = US_ZIP_TO_REGION.get(postal_prefix, "US-Other")
            
    # 3. France Postal Code detection (Open-set)
    if not postal_code and ("france" in country_lower or country_norm == "FR"):
        fr_match = re.search(r"\b(\d{5})\b", combined_addr)
        if fr_match:
            postal_code = fr_match.group(1)
            postal_prefix = postal_code[:2]
            inferred_region = FRANCE_POSTAL_TO_REGION.get(postal_prefix, "France-Other")
            
    # Fallback postal search if country unstated or unmapped
    if not postal_code:
        any_code = re.search(r"\b(\d{5,6})\b", combined_addr)
        if any_code:
            postal_code = any_code.group(1)
            postal_prefix = postal_code[:2]
            inferred_region = "Unknown-Region"

    return {
        "postal_code": postal_code,
        "postal_prefix": postal_prefix,
        "inferred_region": inferred_region
    }

def extract_country_features(addr_norm: str, country_norm: str) -> Dict[str, Any]:
    """Tags country patterns based on address syntax indicators."""
    addr_lower = addr_norm.lower()
    
    # Tag address format patterns
    pattern = "UNKNOWN_FORMAT"
    if any(k in addr_lower for k in ["khasra", "kh number", "nagar", "colony", "marg", "bhopal", "delhi"]):
        pattern = "INDIA_FORMAT"
    elif any(k in addr_lower for k in ["street", "drive", "road", "avenue", "lane", "boulevard", "court"]) and any(st in addr_lower for st in [" nc", " ok", " az", " md", " tx", " ca", " ny", " fl"]):
        pattern = "US_FORMAT"
    elif any(k in addr_lower for k in ["rue", "boulevard", "avenue", "cedex", "paris", "lyon"]):
        pattern = "FRANCE_FORMAT"
        
    return {
        "addr_country_pattern": pattern
    }

def extract_completeness_features(name_norm: str, addr_norm: str, country_norm: str, postal_code: str) -> Dict[str, Any]:
    """Calculates data completeness metrics and sparsity tier."""
    is_name_empty = 1 if not name_norm.strip() else 0
    is_addr_empty = 1 if not addr_norm.strip() else 0
    is_country_empty = 1 if not country_norm.strip() else 0
    is_postal_empty = 1 if not postal_code.strip() else 0
    
    # Total fields = 4 (name, address, country, postal_code)
    populated = (1 - is_name_empty) + (1 - is_addr_empty) + (1 - is_country_empty) + (1 - is_postal_empty)
    score = populated / 4.0
    
    tier = "HIGH" if score >= 0.75 else ("MEDIUM" if score >= 0.50 else "LOW")
    
    return {
        "is_name_empty": is_name_empty,
        "is_addr_empty": is_addr_empty,
        "is_country_empty": is_country_empty,
        "is_postal_empty": is_postal_empty,
        "completeness_score": score,
        "sparsity_tier": tier
    }

def extract_cross_field_features(name_norm: str, addr_norm: str, country_norm: str, postal_code: str, addr_pattern: str) -> Dict[str, Any]:
    """Computes interaction features between fields."""
    # Check if postal pattern matches stated country
    country_postal_match = -1
    c_lower = country_norm.lower()
    if postal_code:
        if ("india" in c_lower and len(postal_code) == 6) or ("us" in c_lower and len(postal_code) == 5):
            country_postal_match = 1
        elif ("india" in c_lower and len(postal_code) == 5) or ("us" in c_lower and len(postal_code) == 6):
            country_postal_match = 0
            
    # Check if significant name tokens appear in address
    name_tokens = [t for t in name_norm.split() if len(t) > 3 and t not in ["inc", "llc", "ltd", "corp", "pvt"]]
    name_in_address = 1 if any(t in addr_norm for t in name_tokens) else 0
    
    # Combined joint representation
    combined_text = f"{name_norm} {addr_norm}".strip()
    
    return {
        "country_postal_match": country_postal_match,
        "name_in_address": name_in_address,
        "combined_text": combined_text
    }

print("All per-record feature extractors defined successfully!")


All per-record feature extractors defined successfully!


## Step 5: Unified Feature Builder (`build_features_for_record`)
Combines all feature extractors into one unified dictionary per record.


In [6]:
def build_features_for_record(row: Dict[str, str]) -> Dict[str, Any]:
    """Orchestrates all Phase 2 feature modules for a single record."""
    entity_id = row["entity_id"]
    source = row.get("source", "")
    
    name_raw = row.get("business_name_raw", "") or ""
    name_norm = row.get("business_name_normalized", "") or ""
    addr_raw = row.get("business_address_raw", "") or ""
    addr_norm = row.get("business_address_normalized", "") or ""
    country_raw = row.get("country_raw", "") or ""
    country_norm = row.get("country_normalized", "") or ""
    
    # 1. Name features
    f_name = extract_name_features(name_norm)
    # 2. Address features
    f_addr = extract_address_features(addr_norm, addr_raw)
    # 3. Geo features
    f_geo = extract_geographic_features(addr_raw, addr_norm, country_norm)
    # 4. Country features
    f_country = extract_country_features(addr_norm, country_norm)
    # 5. Completeness features
    f_comp = extract_completeness_features(name_norm, addr_norm, country_norm, f_geo["postal_code"])
    # 6. Cross-field features
    f_cross = extract_cross_field_features(name_norm, addr_norm, country_norm, f_geo["postal_code"], f_country["addr_country_pattern"])
    
    record = {
        "entity_id": entity_id,
        "source": source,
        "business_name_raw": name_raw,
        "business_name_normalized": name_norm,
        "business_address_raw": addr_raw,
        "business_address_normalized": addr_norm,
        "country_raw": country_raw,
        "country_normalized": country_norm,
        **f_name,
        **f_addr,
        **f_geo,
        **f_country,
        **f_comp,
        **f_cross
    }
    return record

# Quick test on sample record
sample_test_rec = {
    "entity_id": "S1-925783039",
    "source": "source1",
    "business_name_raw": "Orelee's Barbershop Inc",
    "business_name_normalized": "orelee s barbershop inc",
    "business_address_raw": "1795 Westchester Drive, High Point, NC 27262",
    "business_address_normalized": "1795 westchester drive, high point, nc 27262",
    "country_raw": "US",
    "country_normalized": "US"
}

built = build_features_for_record(sample_test_rec)
print("=== FEATURE BUILDER SAMPLE RECORD ===")
for k, v in built.items():
    print(f"  {k:28}: {v}")


=== FEATURE BUILDER SAMPLE RECORD ===
  entity_id                   : S1-925783039
  source                      : source1
  business_name_raw           : Orelee's Barbershop Inc
  business_name_normalized    : orelee s barbershop inc
  business_address_raw        : 1795 Westchester Drive, High Point, NC 27262
  business_address_normalized : 1795 westchester drive, high point, nc 27262
  country_raw                 : US
  country_normalized          : US
  name_char_len               : 23
  name_word_count             : 4
  name_tokens_sorted          : barbershop inc orelee s
  name_first_token            : orelee
  name_last_token             : inc
  name_soundex                : O640
  addr_char_len               : 44
  addr_word_count             : 7
  addr_tokens_sorted          : 1795 27262 drive high nc point westchester
  addr_first_token            : 1795
  addr_has_landmark           : 0
  postal_code                 : 27262
  postal_prefix               : 27
  inferred_regio

## Step 6: Pairwise Similarity Demonstration on Ground Truth Pairs
We evaluate known true matches from `train_ground_truth.tsv` and measure their pairwise similarity using:
- **Jaccard Similarity** (Token Overlap)
- **Levenshtein Similarity** (Edit Distance)
- **TF-IDF Cosine Similarity**


In [7]:
def load_phase1_sample(file_path: Path, max_rows: int = 50000) -> Dict[str, Dict[str, str]]:
    recs = {}
    if not file_path.exists():
        return recs
    with open(file_path, mode="r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for i, row in enumerate(reader):
            recs[row["entity_id"]] = row
            if i + 1 >= max_rows:
                break
    return recs

print("Loading Phase 1 normalized samples for Ground Truth pairwise verification...")
p1_s1 = load_phase1_sample(PHASE1_DIR / "train_source1_normalized.tsv", 50000)
p1_s2 = load_phase1_sample(PHASE1_DIR / "train_source2_normalized.tsv", 50000)
p1_s3 = load_phase1_sample(PHASE1_DIR / "train_source3_normalized.tsv", 50000)

pairwise_evaluations = []
if GT_FILE.exists():
    with open(GT_FILE, mode="r", encoding="utf-8", errors="replace") as f_gt:
        reader = csv.DictReader(f_gt, delimiter="\t")
        for row in reader:
            s1_id = row["source1_entity_id"]
            if s1_id in p1_s1:
                s1_rec = p1_s1[s1_id]
                for match_id in row["matched_entity_ids"].split(","):
                    match_id = match_id.strip()
                    m_rec = p1_s2.get(match_id) or p1_s3.get(match_id)
                    if m_rec:
                        name_a = s1_rec["business_name_normalized"]
                        name_b = m_rec["business_name_normalized"]
                        addr_a = s1_rec["business_address_normalized"]
                        addr_b = m_rec["business_address_normalized"]
                        
                        j_name = jaccard_similarity(name_a, name_b)
                        l_name = levenshtein_similarity(name_a, name_b)
                        t_name = tfidf_cosine_similarity(name_a, name_b)
                        
                        j_addr = jaccard_similarity(addr_a, addr_b)
                        l_addr = levenshtein_similarity(addr_a, addr_b)
                        
                        pairwise_evaluations.append({
                            "s1_id": s1_id,
                            "match_id": match_id,
                            "name_s1": name_a,
                            "name_match": name_b,
                            "jaccard_name": j_name,
                            "levenshtein_name": l_name,
                            "tfidf_cosine_name": t_name,
                            "jaccard_addr": j_addr,
                            "levenshtein_addr": l_addr
                        })
            if len(pairwise_evaluations) >= 5:
                break

print(f"\nEvaluated {len(pairwise_evaluations)} true matching pairs. Displaying results:")
print("=" * 110)
for idx, p in enumerate(pairwise_evaluations, 1):
    print(f"PAIR #{idx} [{p['s1_id']} <--> {p['match_id']}]")
    print(f"  Name 1: '{p['name_s1']}'")
    print(f"  Name 2: '{p['name_match']}'")
    print(f"  --> Jaccard (Tokens): {p['jaccard_name']:.4f} | Levenshtein: {p['levenshtein_name']:.4f} | TF-IDF Cosine: {p['tfidf_cosine_name']:.4f}")
    print(f"  --> Address Jaccard: {p['jaccard_addr']:.4f} | Address Levenshtein: {p['levenshtein_addr']:.4f}")
    print("-" * 110)


Loading Phase 1 normalized samples for Ground Truth pairwise verification...

Evaluated 5 true matching pairs. Displaying results:
PAIR #1 [S1-210849781 <--> S2-845708664]
  Name 1: 'urology partners inc'
  Name 2: 'urology partners inc'
  --> Jaccard (Tokens): 1.0000 | Levenshtein: 1.0000 | TF-IDF Cosine: 1.0000
  --> Address Jaccard: 0.6000 | Address Levenshtein: 0.8333
--------------------------------------------------------------------------------------------------------------
PAIR #2 [S1-523975207 <--> S3-804250913]
  Name 1: 'vl sprott'
  Name 2: 'vl sprott'
  --> Jaccard (Tokens): 1.0000 | Levenshtein: 1.0000 | TF-IDF Cosine: 1.0000
  --> Address Jaccard: 0.5455 | Address Levenshtein: 0.1875
--------------------------------------------------------------------------------------------------------------
PAIR #3 [S1-239320834 <--> S3-51362987]
  Name 1: 'painters local union 634'
  Name 2: 'mirapyra formerly painters local union 634'
  --> Jaccard (Tokens): 0.6667 | Levenshtein: 0.5

## Step 7: Batch Processing & Export to `data/processed/phase2/`
We stream the normalized files from Phase 1 and build enriched feature tables for all 6 files (train and test, sources 1, 2, 3).


In [8]:
def process_feature_file(input_file: Path, output_file: Path, max_rows: Optional[int] = None):
    """Streams a Phase 1 normalized TSV and outputs enriched feature table in Phase 2."""
    print(f"\nExtracting Features: {input_file.name} --> {output_file.name}...")
    processed_count = 0
    
    with open(input_file, mode="r", encoding="utf-8", errors="replace") as fin:
        reader = csv.DictReader(fin, delimiter="\t")
        
        # Read first row to determine full header dynamically
        first_row = next(reader, None)
        if not first_row:
            print(f"Warning: {input_file.name} is empty.")
            return 0
            
        first_featured = build_features_for_record(first_row)
        fieldnames = list(first_featured.keys())
        
        with open(output_file, mode="w", encoding="utf-8", newline="") as fout:
            writer = csv.DictWriter(fout, fieldnames=fieldnames, delimiter="\t")
            writer.writeheader()
            writer.writerow(first_featured)
            processed_count += 1
            
            for row in reader:
                writer.writerow(build_features_for_record(row))
                processed_count += 1
                
                if processed_count % 500000 == 0:
                    print(f"  ...processed {processed_count:,} rows")
                    
                if max_rows and processed_count >= max_rows:
                    break
                    
    print(f"Completed {output_file.name}: {processed_count:,} total feature records written.")
    return processed_count


## Step 8: Execution across All 6 Files (Train & Test)
Set `MAX_ROWS_PER_FILE = 50000` for a fast verification run (~30s), or `None` for full scale.


In [9]:
MAX_ROWS_PER_FILE = 50000  # Set to None for full production run

tasks = [
    # Train files
    (PHASE1_DIR / "train_source1_normalized.tsv", PHASE2_DIR / "train_source1_features.tsv"),
    (PHASE1_DIR / "train_source2_normalized.tsv", PHASE2_DIR / "train_source2_features.tsv"),
    (PHASE1_DIR / "train_source3_normalized.tsv", PHASE2_DIR / "train_source3_features.tsv"),
    # Test files
    (PHASE1_DIR / "test_source1_normalized.tsv", PHASE2_DIR / "test_source1_features.tsv"),
    (PHASE1_DIR / "test_source2_normalized.tsv", PHASE2_DIR / "test_source2_features.tsv"),
    (PHASE1_DIR / "test_source3_normalized.tsv", PHASE2_DIR / "test_source3_features.tsv"),
]

summary_stats = {}
for in_path, out_path in tasks:
    if in_path.exists():
        count = process_feature_file(in_path, out_path, max_rows=MAX_ROWS_PER_FILE)
        summary_stats[in_path.name] = count
    else:
        print(f"Warning: {in_path.name} not found. Ensure Phase 1 has been executed.")

print("\n=== PHASE 2 FEATURE EXTRACTION SUMMARY ===")
for filename, count in summary_stats.items():
    print(f"- {filename} --> {count:,} feature records generated.")



Extracting Features: train_source1_normalized.tsv --> train_source1_features.tsv...
Completed train_source1_features.tsv: 50,000 total feature records written.

Extracting Features: train_source2_normalized.tsv --> train_source2_features.tsv...
Completed train_source2_features.tsv: 50,000 total feature records written.

Extracting Features: train_source3_normalized.tsv --> train_source3_features.tsv...
Completed train_source3_features.tsv: 50,000 total feature records written.

Extracting Features: test_source1_normalized.tsv --> test_source1_features.tsv...
Completed test_source1_features.tsv: 50,000 total feature records written.

Extracting Features: test_source2_normalized.tsv --> test_source2_features.tsv...
Completed test_source2_features.tsv: 50,000 total feature records written.

Extracting Features: test_source3_normalized.tsv --> test_source3_features.tsv...
Completed test_source3_features.tsv: 50,000 total feature records written.

=== PHASE 2 FEATURE EXTRACTION SUMMARY ===

## Step 9: Quality & Integrity Verification Checks
We verify that:
1. All generated feature files exist in `data/processed/phase2/`.
2. All raw and normalized columns from Phase 1 are strictly preserved.
3. Every feature column is populated without errors.


In [10]:
def verify_phase2_outputs(feature_file: Path, max_check: int = 10000):
    """Runs assertions on Phase 2 feature output table."""
    print(f"Verifying feature table: {feature_file.name}...")
    required_cols = [
        "entity_id", "business_name_raw", "business_name_normalized",
        "name_char_len", "name_word_count", "name_tokens_sorted", "name_first_token", "name_soundex",
        "addr_char_len", "addr_has_landmark", "postal_code", "inferred_region",
        "addr_country_pattern", "completeness_score", "sparsity_tier", "combined_text"
    ]
    
    with open(feature_file, mode="r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f, delimiter="\t")
        assert all(c in reader.fieldnames for c in required_cols), f"Missing required columns in {feature_file.name}"
        
        checked = 0
        for row in reader:
            assert row["entity_id"], "Missing entity_id"
            assert float(row["completeness_score"]) >= 0.0, "Invalid completeness score"
            assert "name_soundex" in row and len(row["name_soundex"]) >= 4, "Soundex calculation failed"
            checked += 1
            if checked >= max_check:
                break
                
    print(f" PASSED: {feature_file.name} passed all {len(required_cols)} schema & integrity checks ({checked:,} rows verified)!")

# Run verification check on train_source1_features
verify_phase2_outputs(PHASE2_DIR / "train_source1_features.tsv")


Verifying feature table: train_source1_features.tsv...
 PASSED: train_source1_features.tsv passed all 16 schema & integrity checks (10,000 rows verified)!
